# Notebook to demostrate the new EDAB python utility functions

## Import libraries and load all utility functions

In [2]:
# Import libraries
import os
import xarray as xr
import socket
import sys
from pathlib import Path
from datetime import datetime, date, timedelta
import calendar



In [3]:
current_path = Path.cwd()

# Search for the utilities directory
for parent in current_path.parents:
    candidate = parent / "utilities"
    if candidate.is_dir():
        sys.path.insert(0, str(parent))
        utilities_path = candidate
        break
else:
    raise FileNotFoundError("Could not locate 'utilities' directory in parent paths.")

from utilities import init_notebook_environment
from utilities import function_utilities, get_pyfile_functions, find_function, find_text
from utilities import file_utilities, get_file_dates, file_make, set_file_permissions
from utilities import get_dataset_products,get_prod_files,dataset_defaults,run_psc_pipeline,subset_dataset, parse_dataset_info
from utilities import date_utilities, get_dates,get_source_file_dates
from utilities import mapping_utilities
from utilities import run_stats_pipeline,get_file_dates
from utilities import statanom_utilities, calc_primprod, calc_phytosizeclass
from utilities import period_utilities, period_info, get_period_info, get_period_sets, get_period_dates, extract_period_code, check_period, generate_periods

env = init_notebook_environment(verbose=False)


## Show the environment

In [ ]:
from utilities import bootstrap_environment
e = bootstrap_environment(verbose=False)
e

## Show the default python utilties directory and get the names of the functions

In [ ]:
utilities_path

### Find the names of the python utility functions

In [3]:
# Get the names of the .py files in the utilities folder
from utilities import get_pyfile_functions
get_pyfile_functions()

directory: /Users/kimberly.hyde/Documents/nadata/RESOURCES/python/utilities


{'statanom_utilities': ['get_groupby_hint',
  'apply_grouping',
  'compute_stats',
  'build_stats_map',
  'run_stats_pipeline',
  'get_climatology'],
 'calc_primprod': ['validate_inputs',
  'build_pp_date_map',
  'vgpm_models',
  'process_daily_pp',
  'run_pp_pipeline'],
 'metadata_utilities': ['metadata_dict_to_df',
  'get_metadata_table',
  'load_all_metadata',
  'get_default_metadata',
  'extract_metadata_from_attrs',
  'get_lut_metadata',
  'get_geospatial_metadata',
  'get_temporal_metadata',
  'get_reference_metadata',
  'get_source_metadata',
  'get_lut_products',
  'build_product_attributes',
  'get_fill_value',
  'get_summary_metadata'],
 'calc_mapped_pixel_area': ['pixel_area'],
 'product_utilities': ['product_defaults',
  'netcdf_product_defaults',
  'get_nc_prod',
  'get_prod_files',
  'make_product_output_dir',
  'make_stats_output_dir'],
 'period_utilities': ['get_token_lengths',
  'period_info',
  'get_period_info',
  'check_period',
  'extract_period_code',
  'get_perio

### Search for functions

In [4]:
from utilities import find_function
print,find_function('resolve_dataset_map')


directory: /Users/kimberly.hyde/Documents/nadata/RESOURCES/python/utilities


(<function print(*args, sep=' ', end='\n', file=None, flush=False)>,
 [{'module': 'dataset_utilities', 'function': 'resolve_dataset_map'}])

In [ ]:
#Search for all functions with "get" in the name
print,find_function('get')


In [ ]:
# Search for all functions with "dataset" in the name
print,find_function('dataset')

In [ ]:
# Search for a random string that doesn't exist
find_function('asfasf')

In [13]:
find_function("corrupt_file_detector")

directory: /Users/kimberly.hyde/Documents/nadata/RESOURCES/python/utilities


[{'module': 'file_utilities', 'function': 'corrupt_file_detector'}]

In [12]:
find_text("corrupt")

regex for string corrupt is: re.compile('corrupt', re.IGNORECASE)


[{'module': 'statanom_utilities',
  'function': 'process_single_stat',
  'line_number': 312,
  'line': 'corrupt_files = corrupt_file_detector(input_files)'},
 {'module': 'statanom_utilities',
  'function': 'process_single_stat',
  'line_number': 313,
  'line': 'if corrupt_files:'},
 {'module': 'statanom_utilities',
  'function': 'process_single_stat',
  'line_number': 314,
  'line': 'raise RuntimeError(f"Corrupt files found: {corrupt_files}")'},
 {'module': 'statanom_utilities',
  'function': 'process_single_stat',
  'line_number': 316,
  'line': 'raise RuntimeError(f"Handle Error (No corruption detected): {e}")'},
 {'module': 'statanom_utilities',
  'function': 'run_stats_pipeline',
  'line_number': 617,
  'line': 'all_corrupt_inputs = [] # Track specific source files that are broken'},
 {'module': 'statanom_utilities',
  'function': 'run_stats_pipeline',
  'line_number': 677,
  'line': 'if not failed_months and not all_corrupt_inputs:'},
 {'module': 'statanom_utilities',
  'function'

## Use getfiles.py functions to find datasets and files

In [ ]:
from utilities import get_datasets_source
# Auto-detect the default datasets directory
print(get_datasets_source())

# Manually select 'laptop' datasts directory
print(get_datasets_source(preferred="laptop"))

# Manually select 'server', but default back to laptop if not found
print(get_datasets_source(preferred="server"))

## Show the dataset and product defaults

In [9]:
from utilities import dataset_defaults
dataset_defaults()

{'ACSPO': ('V2.8.1', 'GLOBAL_2KM', 'SST'),
 'ACSPONRT': ('V2.8.1', 'GLOBAL_2KM', 'SST'),
 'AVHRR': ('V5.3', 'GLOBAL_4KM', 'SST'),
 'CORALSST': ('V3.1', 'GLOBAL_5KM', 'SST'),
 'GLOBCOLOUR': ('V4.2.1', 'GLOBAL_4KM', 'CHL'),
 'GLORYS': ('V12', 'GLOBAL_9KM', 'BTEMP'),
 'MUR': ('V4.1', 'GLOBAL_1KM', 'SST'),
 'OCCCI': ('V6.0', 'GLOBAL_4KM', 'CHL'),
 'OISST': ('V2', 'GLOBAL_25KM', 'SST'),
 'PACE': ('V3.1', 'GLOBAL_4KM', 'CHL')}

In [10]:
from utilities import product_defaults
product_defaults()

{'CHL': ('CHL', 'OCCCI', 'SOURCE'),
 'CHLOR_A': ('CHLOR_A', 'OCCCI', 'PRODUCT'),
 'SST': ('SST', 'ACSPO', 'SOURCE'),
 'PPD': ('PPD', 'OCCCI', 'PRODUCT'),
 'PSC': ('PSC', 'OCCCI', 'PRODUCT'),
 'RRS': ('RRS', 'OCCCI', 'SOURCE'),
 'PAR': ('PAR', 'GLOBCOLOUR', 'SOURCE'),
 'IPAR': ('IPAR', 'PACE', 'SOURCE'),
 'AVW': ('AVW', 'OCCCI', 'PRODUCT'),
 'KD': ('KD', 'OCCCI', 'SOURCE'),
 'IOP': ('IOP', 'OCCCI', 'SOURCE'),
 'MOANA': ('MOANA', 'PACE', 'SOURCE'),
 'CARBON': ('CARBON', 'PACE', 'SOURCE'),
 'FLH': ('FLH', 'PACE', 'SOURCE'),
 'CHL_TEMP': ('CHL', 'GLOBCOLOUR', 'SOURCE'),
 'SST_TEMP': ('SST', 'ACSPONRT', 'SOURCE'),
 'CHL_FRONTS': ('CHL_FRONTS', 'OCCCI', 'PRODUCT'),
 'SST_FRONTS': ('SST_FRONTS', 'ACSPO', 'PRODUCT'),
 'FRONTS': ('SST_FRONTS', 'ACSPO', 'PRODUCT')}

In [ ]:
from utilities import netcdf_product_defaults
netcdf_product_defaults()

In [ ]:
from utilities import get_nc_prod
prod = get_nc_prod('CORALSST','SST')
print(prod)

print(get_nc_prod('TESTDATASET', 'SST'))


## Get dataset directories

In [11]:
from utilities import get_dataset_dirs
get_dataset_dirs()

No dataset specified. Returning available datasets from dataset_info_map.


{'GLOBCOLOUR': {'SOURCE': '/Users/kimberly.hyde/Documents/nadata/DATASETS/GLOBCOLOUR/V4.2.1/SOURCE'},
 'OISST': {'SOURCE': '/Users/kimberly.hyde/Documents/nadata/DATASETS/OISST/V2/SOURCE'},
 'PACE': {'SOURCE': '/Users/kimberly.hyde/Documents/nadata/DATASETS/PACE/V3.1/SOURCE'},
 'OCCCI': {'PRODUCTS': '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS',
  'SOURCE': '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/SOURCE'},
 'CORALSST': {'PRODUCTS': '/Users/kimberly.hyde/Documents/nadata/DATASETS/CORALSST/V3.1/PRODUCTS',
  'SOURCE': '/Users/kimberly.hyde/Documents/nadata/DATASETS/CORALSST/V3.1/SOURCE'},
 'MUR': {'SOURCE': '/Users/kimberly.hyde/Documents/nadata/DATASETS/MUR/V4.1/SOURCE'},
 'AVHRR': {'SOURCE': '/Users/kimberly.hyde/Documents/nadata/DATASETS/AVHRR/V5.3/SOURCE'},
 'GLORYS': {'SOURCE': '/Users/kimberly.hyde/Documents/nadata/DATASETS/GLORYS/V12/SOURCE'},
 'ACSPO': {'PRODUCTS': '/Users/kimberly.hyde/Documents/nadata/DATASETS/ACSPO/V2.81/PRODUCTS',
  'S

In [ ]:
from utilities import get_dataset_products
dirs = get_dataset_products('OCCCI')
for source_type, map_types in dirs.items():
    print(f"\n🔷 Source Type: {source_type}")
    for map_type, products in map_types.items():
        print(f"  📦 Map Type: {map_type}")
        for product, path in products.items():
            print(f"    🧪 Product: {product}")
            print(f"      📍 Path: {path}")


### Get product files

In [ ]:
from utilities import get_prod_files
get_prod_files('chl')

In [ ]:
get_prod_files('SST')


In [ ]:
get_prod_files('SST',dataset='CORALSST')

In [ ]:
get_prod_files('PPD')

In [ ]:
xr.open_mfdataset(get_prod_files('chl',dataset='OCCCI'))


In [ ]:
from utilities import get_prod_files

xr.open_mfdataset(get_prod_files('sst',dataset='ACSPO'))

## Date functions

### Filename dates
Extract the dates from the original source file names

In [9]:
cfiles = get_prod_files('chl')
pfiles = get_prod_files('par')

print(get_source_file_dates(cfiles))
print(get_source_file_dates(pfiles))

📦 Found 454 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/SOURCE/GLOBAL_4KM_DAILY/CHL
📦 Found 31 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS/GLOBCOLOUR/V4.2.1/SOURCE/MAPPED_4KM_DAILY/PAR
['19980101', '19980102', '19980103', '19980104', '19980105', '19980106', '19980107', '19980108', '19980109', '19980110', '19980111', '19980112', '19980113', '19980114', '19980115', '19980116', '19980117', '19980118', '19980119', '19980120', '19980121', '19980122', '19980123', '19980124', '19980125', '19980126', '19980127', '19980128', '19980129', '19980130', '19980131', '19980201', '19980202', '19980203', '19980204', '19980205', '19980206', '19980207', '19980208', '19980209', '19980210', '19980211', '19980212', '19980213', '19980214', '19980215', '19980216', '19980217', '19980218', '19980219', '19980220', '19980221', '19980222', '19980223', '19980224', '19980225', '19980226', '19980227', '19980228', '19980301', '19980302', '19980303', '19980304', '19980305', '

### Get dates - Create a list of dates in various output formats
Default output format is YYYYMMDD

In [15]:
get_dates("2025-01-05")

['20250105']

Datetime format

In [11]:
get_dates([2023,2024],day_format='datetime')

[datetime.date(2023, 1, 1),
 datetime.date(2023, 1, 2),
 datetime.date(2023, 1, 3),
 datetime.date(2023, 1, 4),
 datetime.date(2023, 1, 5),
 datetime.date(2023, 1, 6),
 datetime.date(2023, 1, 7),
 datetime.date(2023, 1, 8),
 datetime.date(2023, 1, 9),
 datetime.date(2023, 1, 10),
 datetime.date(2023, 1, 11),
 datetime.date(2023, 1, 12),
 datetime.date(2023, 1, 13),
 datetime.date(2023, 1, 14),
 datetime.date(2023, 1, 15),
 datetime.date(2023, 1, 16),
 datetime.date(2023, 1, 17),
 datetime.date(2023, 1, 18),
 datetime.date(2023, 1, 19),
 datetime.date(2023, 1, 20),
 datetime.date(2023, 1, 21),
 datetime.date(2023, 1, 22),
 datetime.date(2023, 1, 23),
 datetime.date(2023, 1, 24),
 datetime.date(2023, 1, 25),
 datetime.date(2023, 1, 26),
 datetime.date(2023, 1, 27),
 datetime.date(2023, 1, 28),
 datetime.date(2023, 1, 29),
 datetime.date(2023, 1, 30),
 datetime.date(2023, 1, 31),
 datetime.date(2023, 2, 1),
 datetime.date(2023, 2, 2),
 datetime.date(2023, 2, 3),
 datetime.date(2023, 2, 4)

YYYY-MM-DD format

In [16]:
get_dates(['20250505'],day_format='%Y-%m-%d')

['2025-05-05']

Get a list of daily dates within a year

In [12]:
get_dates([2023])

['20230101',
 '20230102',
 '20230103',
 '20230104',
 '20230105',
 '20230106',
 '20230107',
 '20230108',
 '20230109',
 '20230110',
 '20230111',
 '20230112',
 '20230113',
 '20230114',
 '20230115',
 '20230116',
 '20230117',
 '20230118',
 '20230119',
 '20230120',
 '20230121',
 '20230122',
 '20230123',
 '20230124',
 '20230125',
 '20230126',
 '20230127',
 '20230128',
 '20230129',
 '20230130',
 '20230131',
 '20230201',
 '20230202',
 '20230203',
 '20230204',
 '20230205',
 '20230206',
 '20230207',
 '20230208',
 '20230209',
 '20230210',
 '20230211',
 '20230212',
 '20230213',
 '20230214',
 '20230215',
 '20230216',
 '20230217',
 '20230218',
 '20230219',
 '20230220',
 '20230221',
 '20230222',
 '20230223',
 '20230224',
 '20230225',
 '20230226',
 '20230227',
 '20230228',
 '20230301',
 '20230302',
 '20230303',
 '20230304',
 '20230305',
 '20230306',
 '20230307',
 '20230308',
 '20230309',
 '20230310',
 '20230311',
 '20230312',
 '20230313',
 '20230314',
 '20230315',
 '20230316',
 '20230317',
 '20230318',

### Create different temporal outputs (default is daily)
Weekly

In [17]:
get_dates(["19970904","20260302"],output="weeks")

[(1997, 35),
 (1997, 36),
 (1997, 37),
 (1997, 38),
 (1997, 39),
 (1997, 40),
 (1997, 41),
 (1997, 42),
 (1997, 43),
 (1997, 44),
 (1997, 45),
 (1997, 46),
 (1997, 47),
 (1997, 48),
 (1997, 49),
 (1997, 50),
 (1997, 51),
 (1997, 52),
 (1998, 1),
 (1998, 2),
 (1998, 3),
 (1998, 4),
 (1998, 5),
 (1998, 6),
 (1998, 7),
 (1998, 8),
 (1998, 9),
 (1998, 10),
 (1998, 11),
 (1998, 12),
 (1998, 13),
 (1998, 14),
 (1998, 15),
 (1998, 16),
 (1998, 17),
 (1998, 18),
 (1998, 19),
 (1998, 20),
 (1998, 21),
 (1998, 22),
 (1998, 23),
 (1998, 24),
 (1998, 25),
 (1998, 26),
 (1998, 27),
 (1998, 28),
 (1998, 29),
 (1998, 30),
 (1998, 31),
 (1998, 32),
 (1998, 33),
 (1998, 34),
 (1998, 35),
 (1998, 36),
 (1998, 37),
 (1998, 38),
 (1998, 39),
 (1998, 40),
 (1998, 41),
 (1998, 42),
 (1998, 43),
 (1998, 44),
 (1998, 45),
 (1998, 46),
 (1998, 47),
 (1998, 48),
 (1998, 49),
 (1998, 50),
 (1998, 51),
 (1998, 52),
 (1998, 53),
 (1999, 1),
 (1999, 2),
 (1999, 3),
 (1999, 4),
 (1999, 5),
 (1999, 6),
 (1999, 7),
 (

Monthly

In [ ]:
get_dates(["19980101", "19981231"],output="months",period_format="datetime")


Day of year

In [18]:
get_dates(["19970904","20260302"],output="doys",period_format="tuple")

[(1997, 247),
 (1997, 248),
 (1997, 249),
 (1997, 250),
 (1997, 251),
 (1997, 252),
 (1997, 253),
 (1997, 254),
 (1997, 255),
 (1997, 256),
 (1997, 257),
 (1997, 258),
 (1997, 259),
 (1997, 260),
 (1997, 261),
 (1997, 262),
 (1997, 263),
 (1997, 264),
 (1997, 265),
 (1997, 266),
 (1997, 267),
 (1997, 268),
 (1997, 269),
 (1997, 270),
 (1997, 271),
 (1997, 272),
 (1997, 273),
 (1997, 274),
 (1997, 275),
 (1997, 276),
 (1997, 277),
 (1997, 278),
 (1997, 279),
 (1997, 280),
 (1997, 281),
 (1997, 282),
 (1997, 283),
 (1997, 284),
 (1997, 285),
 (1997, 286),
 (1997, 287),
 (1997, 288),
 (1997, 289),
 (1997, 290),
 (1997, 291),
 (1997, 292),
 (1997, 293),
 (1997, 294),
 (1997, 295),
 (1997, 296),
 (1997, 297),
 (1997, 298),
 (1997, 299),
 (1997, 300),
 (1997, 301),
 (1997, 302),
 (1997, 303),
 (1997, 304),
 (1997, 305),
 (1997, 306),
 (1997, 307),
 (1997, 308),
 (1997, 309),
 (1997, 310),
 (1997, 311),
 (1997, 312),
 (1997, 313),
 (1997, 314),
 (1997, 315),
 (1997, 316),
 (1997, 317),
 (1997

"bounds" returns the start and end date for each time period (e.g. seasons)

In [23]:
get_dates(["19970904","20260302"],output="seasons",period_format="bounds")

[('19970701', '19970930'),
 ('19971001', '19971231'),
 ('19980101', '19980331'),
 ('19980401', '19980630'),
 ('19980701', '19980930'),
 ('19981001', '19981231'),
 ('19990101', '19990331'),
 ('19990401', '19990630'),
 ('19990701', '19990930'),
 ('19991001', '19991231'),
 ('20000101', '20000331'),
 ('20000401', '20000630'),
 ('20000701', '20000930'),
 ('20001001', '20001231'),
 ('20010101', '20010331'),
 ('20010401', '20010630'),
 ('20010701', '20010930'),
 ('20011001', '20011231'),
 ('20020101', '20020331'),
 ('20020401', '20020630'),
 ('20020701', '20020930'),
 ('20021001', '20021231'),
 ('20030101', '20030331'),
 ('20030401', '20030630'),
 ('20030701', '20030930'),
 ('20031001', '20031231'),
 ('20040101', '20040331'),
 ('20040401', '20040630'),
 ('20040701', '20040930'),
 ('20041001', '20041231'),
 ('20050101', '20050331'),
 ('20050401', '20050630'),
 ('20050701', '20050930'),
 ('20051001', '20051231'),
 ('20060101', '20060331'),
 ('20060401', '20060630'),
 ('20060701', '20060930'),
 

The start of each year in datetime format

In [24]:
get_dates(["19970904","20260302"],output="years",period_format="datetime")

[datetime.datetime(1997, 1, 1, 0, 0),
 datetime.datetime(1998, 1, 1, 0, 0),
 datetime.datetime(1999, 1, 1, 0, 0),
 datetime.datetime(2000, 1, 1, 0, 0),
 datetime.datetime(2001, 1, 1, 0, 0),
 datetime.datetime(2002, 1, 1, 0, 0),
 datetime.datetime(2003, 1, 1, 0, 0),
 datetime.datetime(2004, 1, 1, 0, 0),
 datetime.datetime(2005, 1, 1, 0, 0),
 datetime.datetime(2006, 1, 1, 0, 0),
 datetime.datetime(2007, 1, 1, 0, 0),
 datetime.datetime(2008, 1, 1, 0, 0),
 datetime.datetime(2009, 1, 1, 0, 0),
 datetime.datetime(2010, 1, 1, 0, 0),
 datetime.datetime(2011, 1, 1, 0, 0),
 datetime.datetime(2012, 1, 1, 0, 0),
 datetime.datetime(2013, 1, 1, 0, 0),
 datetime.datetime(2014, 1, 1, 0, 0),
 datetime.datetime(2015, 1, 1, 0, 0),
 datetime.datetime(2016, 1, 1, 0, 0),
 datetime.datetime(2017, 1, 1, 0, 0),
 datetime.datetime(2018, 1, 1, 0, 0),
 datetime.datetime(2019, 1, 1, 0, 0),
 datetime.datetime(2020, 1, 1, 0, 0),
 datetime.datetime(2021, 1, 1, 0, 0),
 datetime.datetime(2022, 1, 1, 0, 0),
 datetime.da

## PERIODS functions

### Get period code information

#### Get descriptions of the period information

In [38]:
period_info(help=True)

{'0) groupby': 'xarray groupby key used for statistical aggregation',
 '1) input_period_code': 'base period used to compute this period',
 '2) is_range': 'True if the token encodes a user-defined range period',
 '3) is_running_mean': 'True if the period is a running window period',
 '4) is_climatology': 'True if the period is a climatological period',
 '5) iso_duration': 'ISO 8601 duration string',
 '6) groupby_hint': 'xarray expression for grouping (or None)',
 '7) file_format': 'filename pattern for this period (period code plus the date format)',
 '8) date_format': 'description of date format in filename (e.g. YYYY, YYYYMM, YYYYMMDD_YYYYMMDD)',
 '9) n_date_segments': 'number of date segments in the filename (e.g. 1 for YYYYMM, 2 for YYYYMMDD_YYYYMMDD)',
 '10) description': 'human-readable description of the type of data associate with the period'}

#### Get a list of periods and descriptions

In [39]:
info = get_period_info()

print("Available period codes:")
for p, meta in info.items():
    file_format = meta["file_format"]
    description = meta["description"]
    print(f"{p:5}  {file_format:30}  {description}")

Available period codes:
D      D_YYYYMMDD                      Daily
DD     DD_YYYYMMDD_YYYYMMDD            Range of daily data
DOY    DOY_DDD_YYYY_YYYY               Climatological day of year
DOYS   DOYS_YYYY_YYYY                  Combined climatological days of year
D3     D3_YYYYMMDD_YYYYMMDD            Running 3-day
DD3    DD3_YYYYMMDD_YYYYMMDD           Range of running 3-day data
D8     D8_YYYYMMDD_YYYYMMDD            Running 8-day
DD8    DD8_YYYYMMDD_YYYYMMDD           Range of running 8-day data
W      W_YYYYWW                        Weekly (ISO week format)
WW     WW_YYYYWW_YYYYWW                Range of weekly data
WEEK   WEEK_WW_YYYY_YYYY               Climatological ISO week
WEEKS  WEEKS_YYYY_YYYY                 Combined climatological ISO week
M      M_YYYYMM                        Monthly
MM     MM_YYYYMM_YYYYMM                Range of monthly data
MONTH  MONTH_MM_YYYY_YYYY              Climatological month
MONTHS  MONTHS_YYYY_YYYY                Combined climatological

#### Get a description of the period code table

In [40]:
for field, desc in period_info(help=True).items():
    print(f"{field:16}: {desc}")

0) groupby      : xarray groupby key used for statistical aggregation
1) input_period_code: base period used to compute this period
2) is_range     : True if the token encodes a user-defined range period
3) is_running_mean: True if the period is a running window period
4) is_climatology: True if the period is a climatological period
5) iso_duration : ISO 8601 duration string
6) groupby_hint : xarray expression for grouping (or None)
7) file_format  : filename pattern for this period (period code plus the date format)
8) date_format  : description of date format in filename (e.g. YYYY, YYYYMM, YYYYMMDD_YYYYMMDD)
9) n_date_segments: number of date segments in the filename (e.g. 1 for YYYYMM, 2 for YYYYMMDD_YYYYMMDD)
10) description : human-readable description of the type of data associate with the period


#### Get period specific information

In [41]:
get_period_info('D3')

{'groupby': 'day3',
 'input_period_code': 'D',
 'is_range': False,
 'is_running_mean': True,
 'is_climatology': False,
 'iso_duration': 'P3D',
 'groupby_hint': None,
 'file_format': 'D3_YYYYMMDD_YYYYMMDD',
 'date_format': 'YYYYMMDD',
 'n_date_segments': 2,
 'description': 'Running 3-day',
 'regex': re.compile(r'^D3_(?P<seg0>\d{8})_(?P<seg1>\d{8})$', re.UNICODE),
 'file_format_parts': ['YYYYMMDD', 'YYYYMMDD']}

In [42]:
get_period_info('SEASON')

{'groupby': 'season',
 'input_period_code': 'SEA',
 'is_range': False,
 'is_running_mean': False,
 'is_climatology': True,
 'iso_duration': 'P3M',
 'groupby_hint': 'time.dt.season',
 'file_format': 'SEASON_YYYY_YYYY',
 'date_format': 'YYYY',
 'n_date_segments': 2,
 'description': 'Combined climatological season',
 'regex': re.compile(r'^SEASON_(?P<seg0>\d{4})_(?P<seg1>\d{4})$', re.UNICODE),
 'file_format_parts': ['YYYY', 'YYYY']}

In [43]:
get_period_info('DOY')

{'groupby': 'doy',
 'input_period_code': 'D',
 'is_range': False,
 'is_running_mean': False,
 'is_climatology': True,
 'iso_duration': '',
 'groupby_hint': 'time.dt.dayofyear',
 'file_format': 'DOY_DDD_YYYY_YYYY',
 'date_format': 'DOY',
 'n_date_segments': 3,
 'description': 'Climatological day of year',
 'regex': re.compile(r'^DOY_(?P<seg0>\d{3})_(?P<seg1>\d{4})_(?P<seg2>\d{4})$',
            re.UNICODE),
 'file_format_parts': ['DDD', 'YYYY', 'YYYY']}

#### Get all period information

In [44]:
info = get_period_info()
periods = info.keys()
print("Available period codes:")
for p in periods:
    print(f"Period {p}: {info.get(p)}")

Available period codes:
Period D: {'groupby': 'time', 'input_period_code': 'D', 'is_range': False, 'is_running_mean': False, 'is_climatology': False, 'iso_duration': 'P1D', 'groupby_hint': 'time', 'file_format': 'D_YYYYMMDD', 'date_format': 'YYYYMMDD', 'n_date_segments': 1, 'description': 'Daily', 'regex': re.compile('^D_(?P<seg0>\\d{8})$'), 'file_format_parts': ['YYYYMMDD']}
Period DD: {'groupby': 'time', 'input_period_code': 'D', 'is_range': True, 'is_running_mean': False, 'is_climatology': False, 'iso_duration': 'PnD', 'groupby_hint': 'time', 'file_format': 'DD_YYYYMMDD_YYYYMMDD', 'date_format': 'YYYYMMDD', 'n_date_segments': 2, 'description': 'Range of daily data', 'regex': re.compile('^DD_(?P<seg0>\\d{8})_(?P<seg1>\\d{8})$'), 'file_format_parts': ['YYYYMMDD', 'YYYYMMDD']}
Period DOY: {'groupby': 'doy', 'input_period_code': 'D', 'is_range': False, 'is_running_mean': False, 'is_climatology': True, 'iso_duration': '', 'groupby_hint': 'time.dt.dayofyear', 'file_format': 'DOY_DDD_YYYY_

#### Check specific period information fields
Examples
* Is it a climatology period code? 
* Show the regex for each period


In [45]:
info = get_period_info()
periods = info.keys()
for p in periods:
    i = info.get(p)
    print(f"{p} is climatology = {i['is_climatology']}")

D is climatology = False
DD is climatology = False
DOY is climatology = True
DOYS is climatology = True
D3 is climatology = False
DD3 is climatology = False
D8 is climatology = False
DD8 is climatology = False
W is climatology = False
WW is climatology = False
WEEK is climatology = True
WEEKS is climatology = True
M is climatology = False
MM is climatology = False
MONTH is climatology = True
MONTHS is climatology = True
M3 is climatology = False
MM3 is climatology = False
JFM is climatology = False
AMJ is climatology = False
JAS is climatology = False
OND is climatology = False
SEA is climatology = False
SJFM is climatology = True
SAMJ is climatology = True
SJAS is climatology = True
SOND is climatology = True
SEASON is climatology = True
A is climatology = False
AA is climatology = False
ANNUAL is climatology = True
Y is climatology = False
YY is climatology = False
YEAR is climatology = True


In [46]:
info = get_period_info()
periods = info.keys()
for p in periods:
    i = info.get(p)
    print(f"{p} regex = {i['regex']}")

D regex = re.compile('^D_(?P<seg0>\\d{8})$')
DD regex = re.compile('^DD_(?P<seg0>\\d{8})_(?P<seg1>\\d{8})$')
DOY regex = re.compile('^DOY_(?P<seg0>\\d{3})_(?P<seg1>\\d{4})_(?P<seg2>\\d{4})$')
DOYS regex = re.compile('^DOYS_(?P<seg0>\\d{4})_(?P<seg1>\\d{4})$')
D3 regex = re.compile('^D3_(?P<seg0>\\d{8})_(?P<seg1>\\d{8})$')
DD3 regex = re.compile('^DD3_(?P<seg0>\\d{8})_(?P<seg1>\\d{8})$')
D8 regex = re.compile('^D8_(?P<seg0>\\d{8})_(?P<seg1>\\d{8})$')
DD8 regex = re.compile('^DD8_(?P<seg0>\\d{8})_(?P<seg1>\\d{8})$')
W regex = re.compile('^W_(?P<seg0>\\d{6})$')
WW regex = re.compile('^WW_(?P<seg0>\\d{6})_(?P<seg1>\\d{6})$')
WEEK regex = re.compile('^WEEK_(?P<seg0>\\d{2})_(?P<seg1>\\d{4})_(?P<seg2>\\d{4})$')
WEEKS regex = re.compile('^WEEKS_(?P<seg0>\\d{4})_(?P<seg1>\\d{4})$')
M regex = re.compile('^M_(?P<seg0>\\d{6})$')
MM regex = re.compile('^MM_(?P<seg0>\\d{6})_(?P<seg1>\\d{6})$')
MONTH regex = re.compile('^MONTH_(?P<seg0>\\d{2})_(?P<seg1>\\d{4})_(?P<seg2>\\d{4})$')
MONTHS regex = re.co

In [47]:
info = get_period_info()
periods = info.keys()
for p in periods:
    i = info.get(p)
    print(f"{p} period format = {i['file_format']}")

D period format = D_YYYYMMDD
DD period format = DD_YYYYMMDD_YYYYMMDD
DOY period format = DOY_DDD_YYYY_YYYY
DOYS period format = DOYS_YYYY_YYYY
D3 period format = D3_YYYYMMDD_YYYYMMDD
DD3 period format = DD3_YYYYMMDD_YYYYMMDD
D8 period format = D8_YYYYMMDD_YYYYMMDD
DD8 period format = DD8_YYYYMMDD_YYYYMMDD
W period format = W_YYYYWW
WW period format = WW_YYYYWW_YYYYWW
WEEK period format = WEEK_WW_YYYY_YYYY
WEEKS period format = WEEKS_YYYY_YYYY
M period format = M_YYYYMM
MM period format = MM_YYYYMM_YYYYMM
MONTH period format = MONTH_MM_YYYY_YYYY
MONTHS period format = MONTHS_YYYY_YYYY
M3 period format = M3_YYYYMM_YYYYMM
MM3 period format = MM3_YYYYMM_YYYYMM
JFM period format = JFM_YYYY
AMJ period format = AMJ_YYYY
JAS period format = JAS_YYYY
OND period format = OND_YYYY
SEA period format = SEA_YYYY
SJFM period format = SJFM_YYYY_YYYY
SAMJ period format = SAMJ_YYYY_YYYY
SJAS period format = SJAS_YYYY_YYYY
SOND period format = SOND_YYYY_YYYY
SEASON period format = SEASON_YYYY_YYYY
A peri

### Check that the period is valid

In [ ]:
check_period("M_200001")

In [ ]:
check_period("ME_200001")

### Extract the period from a filename
*** Assumes that the filename starts with the period code

In [ ]:
extract_period_code("M_200001-chl.nc")

In [ ]:
extract_period_code("ME_200001-chl.nc") # Invalid period code

### Get the start and end dates for a specific period codes

#### Daily periods
*  D → SINGLE day (YYYYMMDD)
*  DD → a RANGE of days (YYYYMMDD_YYYYMMDD)
*  D3 → a SINGLE 3-day running mean (YYYYMMDD_YYYYMMDD)
*  DD3 → a RANGE of 3-day running means (YYYYMMDD_YYYYMMDD)
*  D8 → a SINGLE 8-day running mean (YYYYMMDD_YYYYMMDD)
*  DD8 → a RANGE of 8-day running means (YYYYMMDD_YYYYMMDD)
*  DOY → a SINGLE CLIMATOLOGY day with the year span (DOY_YYYY_YYY)
*  DOYS → a COMPLETE CLIMATOLOGY series (day 001 to day 365) with the year span (YYYY_YYYY)

In [5]:
pers = [
    "D_20210401",
    "DD_20190101_20191221",
    "D3_20180101_20180103",
    "DD3_19980101_20181231",
    "D8_20030101_20030108",
    "DD8_20110101_20181231",
    "DOY_234_2015_2025",
    "DOYS_2000_2020"]
pds = get_period_dates(pers)

for per, pd in zip(pers, pds):
    print(f"{per} : {pd}")

D_20210401 : ('20210401', '20210401')
DD_20190101_20191221 : ('20190101', '20191221')
D3_20180101_20180103 : ('20180101', '20180103')
DD3_19980101_20181231 : ('19980101', '20181231')
D8_20030101_20030108 : ('20030101', '20030108')
DD8_20110101_20181231 : ('20110101', '20181231')
DOY_234_2015_2025 : ('20150822', '20251231')
DOYS_2000_2020 : ('20000101', '20201231')


#### Weekly periods
Weekly period are based on the standardized ISO week format. Week 1 of the 7-day, Monday-to-Sunday, week-based calendar is defined as the week containing the first Thursday of the year, usually containing January 4th. Thus week one of the new calendar year may start in December of the preceeding year and week 52/53 may end in January of the following year.
*  W → a SINGLE ISO week (YYYYWW) 
*  WW → a RANGE of ISO weeks (YYYYWW_YYYYWW)
*  WEEK → a SINGLE CLIMATOLOGY of a week with the year span (WW_YYYY_YYYY)
*  WEEKS → a COMPLETE CLIMATOLOGY series (week 1 to 52) with the year span (WW_YYYY_YYYY)

In [6]:
pers = [
    "W_201201",
    "WW_202101_202152",
    "WEEK_03_2006_2020",
    "WEEKS_2000_2020"
    ]
pds = get_period_dates(pers)

for per, pd in zip(pers, pds):
    print(f"{per} : {pd}")

W_201201 : ('20120102', '20120108')
WW_202101_202152 : ('20210104', '20220102')
WEEK_03_2006_2020 : ('20060116', '20200119')
WEEKS_2000_2020 : ('20000101', '20201231')


#### Monthly periods
Daily (D) data are used for the base input for the monthly (M) period
*  M → SINGLE month (YYYYMM)
*  MM → a RANGE of months (YYYYMM_YYYYMM)
*  M3 → a SINGLE 3-month running mean (YYYYMM_YYYYMM)
*  MM3 → a RANGE of 3-month running means (YYYYMM_YYYYMM)
*  MONTH → a SINGLE CLIMATOLOGY month with the year span (MM_YYYY_YYY)
*  MONTHS → a COMPLETE CLIMATOLOGY series (month 1 to month 12) with the year span (YYYY_YYYY)

In [ ]:
pers = [
    "M_201704",
    "MM_201101_201112",
    "M3_201301_201303",
    "MM3_202101_202112",
    "MONTH_11_2000_2020",
    "MONTHS_2000_2020"
    ]
pds = get_period_dates(pers)

for per, pd in zip(pers, pds):
    print(f"{per} : {pd}")

#### Seasonal periods
Monthly (M) data are used for the base input for the seasonal periods. For seasons starting December, January February, use the M3 period
*  JFM → a SINGLE January, February, March season for a year (YYYY)
*  AMJ → a SINGLE April, May, June season for a year (YYYY)
*  JAS → a SINGLE July, August, September season for a year (YYYY)
*  OND → a SINGLE October, November, December season for a year (YYYY)
*  SEA → All seasons for a SINGLE year combined (YYYY)
*  SJFM → The January, February, March seasonal CLIMATOLOGY with the year span (YYYY_YYYY)
*  SAMJ → The April, May, June seasonal CLIMATOLOGY with the year span (YYYY_YYYY)
*  SJAS → The July, August, September seasonal CLIMATOLOGY with the year span (YYYY_YYYY)
*  SOND → The October, November, December seasonal CLIMATOLOGY with the year span (YYYY_YYYY)
*  SEASON → The COMPLETE CLIMATOLOGY for all seasons with a year span (YYYY_YYYY)

In [ ]:
pers = [
    "JFM_2020",
    "AMJ_2020",
    "JAS_2020",
    "OND_2020",
    "SEA_2020",
    "SJFM_2000_2020",
    "SAMJ_2000_2020",
    "SJAS_2000_2020",
    "SOND_2000_2020",
    "SEASON_2000_2020"
    ]
pds = get_period_dates(pers)

for per, pd in zip(pers, pds):
    print(f"{per} : {pd}")

#### Annual periods
Monthly (M) data are used for the base input for the annual (A) period
*  A → SINGLE year (YYYY) (the mean of 12 months)
*  AA → a RANGE of years (YYYY_YYYY)
*  ANNUAL → an annual CLIMATOLOGY with the year span (YYYY_YYYY)

In [ ]:
pers = [
    "A_2020",
    "AA_2010_2025",
    "ANNUAL_1990_2020"
    ]
pds = get_period_dates(pers)

for per, pd in zip(pers, pds):
    print(f"{per} : {pd}")

#### Yearly periods
Daily (D) data are used for the base input for the annual (Y) period
*  Y → SINGLE year (YYYY) (the mean of 12 months)
*  YY → a RANGE of years (YYYY_YYYY)
*  YEAR → a year-based CLIMATOLOGY with the year span (YYYY_YYYY)

In [ ]:
pers = [
    "Y_2020",
    "YY_2010_2025",
    "YEAR_1990_2020"
]

pds = get_period_dates(pers)

for per, pd in zip(pers, pds):
    print(f"{per} : {pd}")

#### Examples of incorrectly formatted periods
1. get_period_dates() uses check_period() to make sure:
* It is a recognized period code 
* The number of segments within the period are correct
* The length of the segments is correct
2. get_period_dates() then checks that:
* Specified dates are within expected ranges
    * Day ranges are based on the number of days in the specified month
    * Day of year ranges are 1 to 366
    * Months ranges are 1 to 12
    * Weeks ranges are 1 and 53 (note some years have 53 weeks)
    * Running means are 3 or 8 days/months long 
    * Seasons are 3 months long and start and end on the correct JFM, AMJ, JAS, OND months
* The start date is sooner than the end date 
All incorrect periods return NA for the start and end date. 
Use diagnostics=True to retrieve the error.

In [ ]:
pers = [
    "BAD_2000_2020", # Invalid period code
    "D_20210491", # Invalid day
    "DD_20190101_20191955", # Invalid month and day
    "D3_20180101_20180104", # Greater than 3 days
    "DD3_19980141_20182231", # Invalid day and month
    "D8_20030105_20030109", # Greater than 8 days
    "DD8_20112101_20181233", # Invalid day and month
    "DOY_443_2010_2025", "DOY_334_2015", # Invalid doy (> 365) & Invalid day of year
    "DOYS_2002002_2022002", # Invalid format"
    "W_201291", # Invalid week (> 52)
    "WW_202101_202166", # Invalid week
    "WEEK_88_2006_2020", # Invalid week (> 88)
    "WEEKS_200011_202011", # Invalid format
    "M_201718", # Invalid month (> 12)
    "MM_201105_201102", #Start month later than end date 
    "M3_201301_201304", # Greater than 3 months
    "MM3_202101_202012", # Start year before end year
    "MONTH_00_2000_2020", # Invalid month (no 00 month)
    "MONTHS_200001_202012", # Invalid format
    "JFM_202001", # Incorrect format
    "AMJ_202", # Invalid year
    "JAM_2020", # Invalid period code
    "ONDS_2020", # Invalid period code
    "SEA_2020_2021", # Incorrect format
    "JFMS_2000_2020", # Invalid period code
    "SAMJ_200004_202006", # Incorrect format
    "SJAS_2000", # Incorrect format
    "OND_2000_2020", # Incorrect period code
    "SEASON_200001_202012", # Incorrect format
    "A_2020_2022", # Incorrect format
    "AA_201001_202512", # Incorrect format
    "ANNUAL_1990", #Incorrect format
    "Y_202", # Invalid format
    "YY_2010_2005", # Start year later than end year
    "YEARS_1990_2020", # Invalid period code
]
pds,errs = get_period_dates(pers,diagnostics=True)

error_map = {fname: msg for fname, msg in errs}

for per, pd in zip(pers, pds):
    err = error_map.get(per, "OK")
    print(f"{per:<25} → {pd}, {err}")

#### Now also works with input file names
*** Assumes the file name starts with the period code

In [ ]:
get_period_dates(["M_200001-chl.nc","M_200002-sst.nc"])

### Create periods from a given date range

In [4]:
start = "19970901"
end = "20260309"
climatology_range = [1991,2020]
d = generate_periods('D',start,end)
dd = generate_periods('DD',start,end)
ddy = generate_periods('DD',start,end,range_by_year=True)
d3 = generate_periods('D3',start,end)
d8 = generate_periods('D8',start,end)
dd3 = generate_periods('DD3',start,end)
dd8 = generate_periods('DD8',start,end)
dd3y = generate_periods('DD3',start,end,range_by_year=True)
dd8y = generate_periods('DD8',start,end,range_by_year=True)
doy = generate_periods('DOY',start,end,climatology_range=climatology_range)
doys = generate_periods('DOYS',start,end,climatology_range=climatology_range)
w = generate_periods('W',start,end)
ww = generate_periods('WW',start,end)
wwy = generate_periods('WW',start,end,range_by_year=True)
week = generate_periods('WEEK',start,end,climatology_range=climatology_range) 
weeks = generate_periods('WEEKS',start,end,climatology_range=climatology_range) 
m = generate_periods('M',start,end)
mm = generate_periods('MM',start,end)
mmy = generate_periods('MM',start,end,range_by_year=True)
m3 = generate_periods('M3',start,end) 
mm3 = generate_periods('MM3',start,end) 
mm3y = generate_periods('MM3',start,end,range_by_year=True)
month = generate_periods('MONTH',start,end,climatology_range=climatology_range) 
months = generate_periods('MONTHS',start,end,climatology_range=climatology_range)  
jfm = generate_periods('JFM',start,end)
amj = generate_periods('AMJ',start,end)
jas = generate_periods('JAS',start,end)
ond = generate_periods('OND',start,end)
sea = generate_periods('SEA',start,end)
season = generate_periods('SEASON',start,end,climatology_range=climatology_range)
a = generate_periods('A',start,end)
annual = generate_periods('ANNUAL',start,end,climatology_range=climatology_range)
y = generate_periods('Y',start,end)
year = generate_periods('YEAR',start,end,climatology_range=climatology_range)

### Create sets of periods based on the input period "files" from above

In [ ]:
d3_perset = get_period_sets('D3',files=d,range_by_year=True)
dd3_perset = get_period_sets('DD3',files=d3)
dd3y_perset = get_period_sets('DD3',files=d3,range_by_year=True)
d8_perset = get_period_sets('D8',files=d,range_by_year=True)
dd8_perset = get_period_sets('DD8',files=d8)
dd8y_perset = get_period_sets('DD8',files=d8,range_by_year=True)
doy_perset = get_period_sets('DOY',files=d)
doys_perset = get_period_sets('DOYS',files=doy)

w_perset = get_period_sets('W',files=d)
ww_perset = get_period_sets('WW',files=w)
wwy_perset = get_period_sets('WW',files=w,range_by_year=True)
week_perset = get_period_sets('WEEK',files=w)
weeks_perset = get_period_sets('WEEKS',files=week)

m_perset = get_period_sets('M',files=d)
mm_perset = get_period_sets('MM',files=m)
mmy_perset = get_period_sets('MM',files=m,range_by_year=True)
m3_perset = get_period_sets('M3',files=m)
mm3_perset = get_period_sets('MM3',files=m3)
mm3y_perset = get_period_sets('MM3',files=m3,range_by_year=True)
month_perset = get_period_sets('MONTH',files=m)
months_perset = get_period_sets('MONTHS',files=month)

jfm_perset = get_period_sets('JFM',files=m)
amj_perset = get_period_sets('AMJ',files=m)
jas_perset = get_period_sets('JAS',files=m)
ond_perset = get_period_sets('OND',files=m)
sea_perset = get_period_sets('SEA',files=m)
season_perset = get_period_sets('SEASON',files=sea)

a_perset = get_period_sets('A',files=m)
annual_perset = get_period_sets('ANNUAL',files=a)
y_perset = get_period_sets('Y',files=d)
year_perset = get_period_sets('YEAR',files=y)


### Period Examples

#### Date range input

In [ ]:
get_period_sets('M',start="19970904",end="19981231")

#### File name input

In [6]:
get_period_sets('A', files=["M_200001-chl.nc", "M_200002-chl.nc",
                            "M_200003-chl.nc", "M_200004-chl.nc",
                            "M_200005-chl.nc", "M_200006-chl.nc",
                            "M_200007-chl.nc", "M_200008-chl.nc",
                            "M_200009-chl.nc", "M_200010-chl.nc",
                            "M_200011-chl.nc", "M_200012-chl.nc"])

{'target_code': 'A',
 'expected_input_code': 'M',
 'start_dt': '20000101',
 'end_dt': '20001231',
 'period_map': {'A_2000': {'input_periods': ['M_200001-chl.nc',
    'M_200002-chl.nc',
    'M_200003-chl.nc',
    'M_200004-chl.nc',
    'M_200005-chl.nc',
    'M_200006-chl.nc',
    'M_200007-chl.nc',
    'M_200008-chl.nc',
    'M_200009-chl.nc',
    'M_200010-chl.nc',
    'M_200011-chl.nc',
    'M_200012-chl.nc'],
   'input_files': ['M_200001-chl.nc',
    'M_200002-chl.nc',
    'M_200003-chl.nc',
    'M_200004-chl.nc',
    'M_200005-chl.nc',
    'M_200006-chl.nc',
    'M_200007-chl.nc',
    'M_200008-chl.nc',
    'M_200009-chl.nc',
    'M_200010-chl.nc',
    'M_200011-chl.nc',
    'M_200012-chl.nc']}},
 'diagnostics': []}

In [5]:
files = get_prod_files('CHL',period='M',map_region='NES')
file = files[0]
extract_period_code(Path(file).name)

📦 Found 33 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_STATS/CHL


('M', 'M_199801', ['199801'], [1998])

#### Range example:
WW is a "range" period that will either put all W periods into a single group or year groups

In [ ]:
start = "19970901"
end = "20260309"
get_period_sets('WW',files=generate_periods('W',start,end),range_by_year=False) 

In [ ]:
start = "19970901"
end = "20260309"
get_period_sets('WW',files=generate_periods('W',start,end),range_by_year=True) 

#### Climatology example:
MONTH is the monthly climatology. The default climatological range is 1991 to 2020

In [ ]:
start = "1990101"
end = "20260309"
get_period_sets('MONTH',files=generate_periods('M',start,end))

In [ ]:
start = "20000301"
end = "20260309"
climatology_range = [2000,2025]
get_period_sets('MONTH',files=generate_periods('M',start,end))

#### The SEASON period contains the climatological season periods (SJFM, SAMJ, SJAS, SOND) and is a special case that can not be created using get_period_sets()

In [ ]:

sea = generate_periods('SEA',start,end)
get_period_sets('SEASON',files=sea)

#### Fails because the climatology DOYS code expects DOY inputs

In [ ]:
dates = get_dates(["19970904","20260302"])
get_period_sets('DOYS',dates=dates)

#### The annual (A) fails because A periods need 12 months

In [ ]:
get_period_sets('A', files=["M_200001", "M_202012",'M_202112','M_202305'])

In [ ]:
def format_period_token(target_code, period_tuple, period_map):
    """
    Format an output period token using the file_format template
    from period_info().
    """
    info = period_map[target_code]
    template = info['file_format']  # e.g., "M_YYYYMM"

    year = period_tuple[0]

    # Monthly periods
    if info['span_type'] == 'month':
        month = period_tuple[1]
        token = template.replace("YYYY", f"{year:04d}") \
                        .replace("MM", f"{month:02d}")
        return token

    # Other span types will be added later
    raise NotImplementedError(f"Span type '{info['span_type']}' not yet implemented.")

In [ ]:
period_map = get_period_info()
format_period_token("M", (2020, 2), period_map)

In [ ]:
def map_inputs_to_month(period_tuple, spans):
    """
    Given (year, month) and a list of (sdt, edt, token),
    return the tokens whose date spans fall within that month.
    """
    year, month = period_tuple

    # Compute the month start and end
    month_start = datetime(year, month, 1)
    if month == 12:
        month_end = datetime(year + 1, 1, 1) - timedelta(days=1)
    else:
        month_end = datetime(year, month + 1, 1) - timedelta(days=1)

    matched = []

    for sdt, edt, token in spans:
        s = datetime.strptime(sdt, "%Y%m%d")
        e = datetime.strptime(edt, "%Y%m%d")

        # Overlap test
        if e >= month_start and s <= month_end:
            matched.append(token)

    return matched

In [ ]:
period_map = get_period_info()
target_code='M'
info = period_map[target_code]
start_dt = "20020101"
end_dt = "20201231"

if info['span_type'] == 'month':
    periods = generate_months(start_dt, end_dt)
    period_map_out = {}

    for p in periods:
        token = format_period_token(target_code, p, period_map)
        inputs = map_inputs_to_month(p, spans)
        period_map_out[token] = {"inputs": inputs}

In [ ]:
get_period_info('M')

In [ ]:
get_period_sets('A', files=["M_200001-chl.nc", "W_202002-chl.nc"])

In [ ]:
get_period_sets('M', files=["E_200001-chl.nc", "E_202002-chl.nc"])

In [ ]:
get_period_sets('M', dates=get_dates([2020,2021]))

In [ ]:
get_period_sets('A', dates=get_dates([2020,2021]))

In [ ]:
get_period_sets('M', dates=get_dates([2020,2021]))

In [ ]:
get_period_sets('M',start="202011",end="20201231")

In [ ]:
get_period_dates(['D_20220101','D_20220102','D_20220103'])


In [ ]:
get_period_sets('M',dates=get_dates([2020,2021]))


## Stats Demo

In [27]:
find_text("stat")

regex for string stat is: re.compile('stat', re.IGNORECASE)


[{'module': 'statanom_utilities',
  'function': 'get_groupby_hint',
  'line_number': 37,
  'line': 'Returns the appropriate xarray groupby hint or rolling strategy for a given statistical period code.'},
 {'module': 'statanom_utilities',
  'function': 'get_groupby_hint',
  'line_number': 41,
  'line': "period_code (str): The code representing the statistical period (e.g. 'D', 'W', 'M3', 'SEASON')."},
 {'module': 'statanom_utilities',
  'function': 'compute_stats',
  'line_number': 88,
  'line': 'def compute_stats(grouped, var_name, time_dim, label):'},
 {'module': 'statanom_utilities',
  'function': 'stats_period_map',
  'line_number': 99,
  'line': 'def stats_period_map(prod, period, dataset=None, dates=None, version=None, dataset_map=None, subset=None, is_running=True, verbose=False):'},
 {'module': 'statanom_utilities',
  'function': 'stats_period_map',
  'line_number': 101,
  'line': 'Constructs a stats period file path map using get_prod_files.'},
 {'module': 'statanom_utilities',

## Regrid Demo

In [ ]:
from utilities import get_prod_files
chl_path = get_prod_files('chl',dataset='OCCCI',getfilepath=True)
sst_path = get_prod_files('sst',dataset='CORALSST',getfilepath=True)
par_path = get_prod_files('par',dataset='GLOBCOLOUR',getfilepath=True)

from utilities import regrid_wrapper
sst_ds = regrid_wrapper(chl_path,sst_path,source_vars=['analysed_sst'])
par_ds = regrid_wrapper(chl_path,par_path,source_vars=['PAR_mean'])
chl_ds = xr.open_mfdataset(os.path.join(chl_path,"*.nc"))


In [ ]:
def compare_grids(ds1, ds2, tol=1e-6):
    lat_match = np.allclose(ds1["lat"].values, ds2["lat"].values, atol=tol)
    lon_match = np.allclose(ds1["lon"].values, ds2["lon"].values, atol=tol)
    return lat_match and lon_match

print("✅ SST grid matches CHL:", compare_grids(sst_ds, chl_ds))
print("✅ PAR grid matches CHL:", compare_grids(par_ds, chl_ds))

def grid_summary(ds, name):
    print(f"📦 {name} grid:")
    print(f"  lat: shape={ds.lat.shape}, range=({ds.lat.min().values:.2f}, {ds.lat.max().values:.2f})")
    print(f"  lon: shape={ds.lon.shape}, range=({ds.lon.min().values:.2f}, {ds.lon.max().values:.2f})")

grid_summary(chl_ds, "CHL")
grid_summary(sst_ds, "SST")
grid_summary(par_ds, "PAR")

### Daylength calculation

In [ ]:
from utilities import get_daylength_grid
from utilities import get_prod_files
chl_path = get_prod_files('chl',dataset='OCCCI',getfilepath=True)

dl = get_daylength_grid(chl_path)
dl

In [ ]:
# Old steps to do a global import of all functions


def global_import(function_map,verbose=False):
    for module_name, function_names in function_map.items():
        try:
            module = importlib.import_module(module_name)
        except ModuleNotFoundError:
            print(f"❌ Module '{module_name}' could not be imported.")
            continue

        for name in function_names:
            try:
                func = getattr(module, name)
                globals()[name] = func
                print(f"✅ Imported: {name} from {module_name}")
            except AttributeError:
                print(f"⚠ Function '{name}' not found in '{module_name}'")
    

# Get the path to the utilities folder
util_path = get_python_path()

# Add to sys.path
if util_path not in sys.path:
    sys.path.append(util_path)

# Find the default utiltity functions and import into global
from import_utilities import get_pyfile_functions
functions = get_pyfile_functions()
gl = global_import(functions)

# Import local calc functions
calc_functions = get_pyfile_functions(os.getcwd(),"calc")
gl = global_import(calc_functions)



In [ ]:
# Old code to find the python path

def get_python_path():
    hostname = socket.gethostname()                                 # 1. Identify the computer by hostname
    code_locations = {                                              # 2. Set default Python code location based on hostname
        "NECMAC04363461.local": "/Users/kimberly.hyde/Documents/",  # Mac laptop
        "nefscsatdata": "/mnt/EDAB_Archive/",                       # Satdata
        "guihyde": "/mnt/EDAB_Archive/"                             # Kim's Satdata container
    }

    base_path = code_locations.get(hostname)
    if not base_path:
        print(f"Unknown hostname: {hostname}")
        return None

    default_utility_path = Path(base_path) / "nadata/python"
    if not default_utility_path.is_dir():
        print(f"Directory not found: {default_utility_path}")
        return None

    print(f"Default utilities path: {default_utility_path}")
    return default_utility_path
